# Demo: Building a Text Generation Pipeline with LangChain and Hugging Face's Flan T5 Large Model

In this demo, you will learn how to create a Langchain HuggingFacePipeline for efficient text generation and dive into the creation of a Langchain chain to craft context-aware responses using a custom template.

#### Step 1: Set up the environment and install the required libraries and dependencies

* The Terminal will open in a new tab. Enter the following command inside the terminal to create a virtual environment

python3 -m venv .venv

.\.venv\Scripts\Activate.ps1

pip install ipykernel

python -m ipykernel install --user --name myenv --display-name "Python (myenv)"

pip install --upgrade pip setuptools wheel

#### Step 2: Import all the required libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

# import os
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = input("Enter your Hugging Face API Key: ")

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from transformers import pipeline

from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
# old version
# from langchain_community.llms import HuggingFacePipeline 
# from langchain.chains import LLMChain 
# from langchain.prompts import PromptTemplate

2.7.0+cu128
True
12.8
NVIDIA GeForce RTX 3080 Ti


#### Step 3: Use the Hugging Face Hub to Load the model

In [2]:
import os

device = 0 if torch.cuda.is_available() else -1

# This environment has a dead local proxy configured, so load from the local HF cache.
for proxy_var in ["HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"]:
    os.environ.pop(proxy_var, None)

hf_pipeline = pipeline(
    "text2text-generation",
    model='google/flan-t5-large',
    device=device,
    max_new_tokens=100,
    torch_dtype=torch.float32,  # Uses lower precision for efficiency 
    model_kwargs={"local_files_only": True},
)

#### Step 4: Create a Langchain HuggingFacePipeline for Text Generation

In [3]:
# Adjust generation parameters
hf_pipeline.model.generation_config.max_new_tokens = 512
hf_pipeline.model.generation_config.temperature = 0.7
hf_pipeline.model.generation_config.do_sample = True #控制文本是否使用采样法

# Wrap in LangChain interface
llm = HuggingFacePipeline(pipeline=hf_pipeline)

#### Step 5: Build a Chain Using Langchain

In [4]:
# ===========================
# Build Prompt + Chain
# ===========================

# template = """
# Question: {question}

# Answer: Let's think step by step.
# """

# prompt = PromptTemplate(
#     template=template,
#     input_variables=["question"]
# )

# from langchain_core.runnables import RunnableLambda
# llm_chain = prompt | llm | RunnableLambda(lambda text: {"text": text})
# chain = llm_chain

In [5]:
template = """
Question: {question}

Answer: Let's think step by step.
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"]
)

llm_chain = prompt | llm
chain = llm_chain

#### Step 6: Test and Run the Chain on Few Questions

In [6]:
# Question 1
question1 = "Explain the concept of black holes in simple terms."
answer1 = llm_chain.invoke({"question": question1})
print("Answer 1:\n", answer1, "\n")

Answer 1:
 Black holes are dark structures that are void of matter. The mass of the supermassive black hole is the same as that of the Sun. So, black holes are dark, and are very large. 



In [8]:
# Question 2
question2 = "Provide a brief overview of the history of artificial intelligence."
answer2 = llm_chain.invoke({"question": question2})
print("Answer 2:\n", answer2, "\n")

Answer 2:
 Artificial intelligence has been a major driver of the development of robotics and automation. The earliest applications of artificial intelligence started in the late 19th century. Robots and automation have greatly accelerated the development of robotics and automation. Robots are a significant contributor to the development of artificial intelligence. 

